In [260]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
from keras import Sequential
from keras.layers import InputLayer, Dense
from keras.utils import to_categorical
from keras.losses import categorical_crossentropy

In [261]:
tf.keras.utils.set_random_seed(42)

dataset = pd.read_csv("fruit.csv")
dataset.head()

,Unnamed: 0,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,0
1,1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,0
2,2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,0
3,3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,0
4,4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,0


In [262]:
labels = dataset.pop("Class")

if "Unnamed: 0" in dataset.columns:
  dataset = dataset.drop(columns = ["Unnamed: 0"])

In [263]:
num_of_fruits = len(labels.unique())
num_of_fruits

7

In [264]:
labels_cat = to_categorical(labels, num_classes = num_of_fruits)

In [265]:
train_dataset, test_dataset, train_labels, test_labels = train_test_split(dataset, labels_cat, test_size = 0.2, random_state = 42, stratify = labels)
train_dataset, validation_dataset, train_labels, validation_labels = train_test_split(train_dataset, train_labels, test_size = 0.2, random_state = 42, stratify = np.argmax(train_labels, axis = 1))

In [266]:
train_stats = train_dataset.describe()
train_stats = train_stats.transpose()

In [267]:
def norm(X):
  return (X - train_stats["mean"]) / train_stats["std"]

In [268]:
normed_train_dataset = norm(train_dataset).astype("float32")
normed_validation_dataset = norm(validation_dataset).astype("float32")
normed_test_dataset = norm(test_dataset).astype("float32")

In [269]:
num_of_atributtes = normed_train_dataset.shape[1]

In [270]:
def build_model(learning_rate):
  model = keras.Sequential([
      InputLayer(input_shape = (num_of_atributtes, )),
      Dense(num_of_atributtes, activation = 'relu'),
      Dense(32, activation = 'relu'),
      Dense(10, activation = 'relu'),
      Dense(num_of_fruits, activation = 'softmax')
  ])

  model.compile(optimizer = keras.optimizers.Adam(learning_rate), loss = categorical_crossentropy, metrics = ["accuracy"])

  return model

In [271]:
learning_rates = [0.001, 0.01, 0.1]

results = []
for learning_rate in learning_rates:
  tf.keras.backend.clear_session()
  tf.keras.utils.set_random_seed(42)

  model = build_model(learning_rate)

  history = model.fit(normed_train_dataset, train_labels, epochs = 30, validation_data = (normed_validation_dataset, validation_labels), verbose = 0)

  best_validation_accuracy = max(history.history["accuracy"])

  results.append({
      "best_validation_accuracy": best_validation_accuracy,
      "learning_rate": learning_rate
  })

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [272]:
# results = pd.DataFrame(results)
results

[{'best_validation_accuracy': 0.9442508816719055, 'learning_rate': 0.001},
 {'best_validation_accuracy': 0.9825783967971802, 'learning_rate': 0.01},
 {'best_validation_accuracy': 0.8362369537353516, 'learning_rate': 0.1}]

In [273]:
best_result = sorted(results, key = lambda result: (-result["best_validation_accuracy"], result["learning_rate"]))[0]
best_learning_rate = best_result["learning_rate"]

In [274]:
train_dataset_full = pd.concat([train_dataset, validation_dataset])
train_labels_full = np.concatenate([train_labels, validation_labels])

In [275]:
train_stats = train_dataset_full.describe()
train_stats = train_stats.transpose()

In [276]:
normed_train_data_full = norm(train_dataset_full).astype("float32")
normed_test_dataset = norm(test_dataset).astype("float32")

In [277]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

In [278]:
model = build_model(best_learning_rate)
history = model.fit(normed_train_data_full, train_labels_full, epochs = 30, verbose = 0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [279]:
test_loss, test_accuracy = model.evaluate(normed_test_dataset, test_labels, verbose = 0)
print(test_loss)
print(test_accuracy)

0.34889882802963257
0.8999999761581421
